<a href="https://colab.research.google.com/github/wbandabarragan/EPIC_6/blob/main/EPIC_Junior/Day_1/2_analisis_de_datos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Análisis del Péndulo Simple: ¿Cuánto es la aceleración de la gravedad, g?

En el experimento anterior midieron el período de oscilación de un péndulo para distintas longitudes. En este notebook vamos a:

1. Subir esa tabla de datos a Google Colab.
2. Graficar los resultados.
3. Usar la física del péndulo para calcular experimentalmente el valor de la gravedad **g**.

Recordemos la fórmula del período de un péndulo simple (válida para ángulos pequeños):

$$T = 2\pi \sqrt{\frac{L}{g}}$$

donde:
- $T$ = período (segundos)
- $L$ = longitud del péndulo (metros)
- $g$ = aceleración de la gravedad (m/s²)

## Paso 1️⃣: Sube tu archivo de datos

Realiza el experimento en el siguiente applet interactivo:

🔗 **https://cphysplus.github.io/fisica-interactiva/pendulo_simple.html**

1. En el applet, ajusta la longitud, suelta el péndulo, inicia el cronómetro y detenlo tú mismo al completar una oscilación completa. Repite para varias longitudes (idealmente varias veces por longitud).
2. Presiona **"Exportar tabla a Excel"**. Esto descargará un archivo llamado `datos_pendulo_simple.xlsx` a tu computadora (revisa tu carpeta de Descargas).
3. Ejecuta la celda de código de abajo (presiona el botón ▶️ a la izquierda de la celda).
4. Va a aparecer un botón **"Elegir archivos"** — selecciona el archivo que acabas de descargar.

> 💡 Si mediste varias longitudes en distintos momentos y tienes más de un archivo, puedes seleccionarlos todos a la vez cuando se abra la ventana de carga: los uniremos automáticamente en una sola tabla.

In [ ]:
# Instalamos y cargamos las librerías necesarias (solo toma unos segundos)
!pip install -q openpyxl

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from google.colab import files

print("Librerías listas ✅")

In [ ]:
print("Selecciona el archivo que descargaste del simulador del péndulo (.xlsx o .csv):")
subido = files.upload()

In [ ]:
# Leemos el archivo (o archivos) que subiste y los combinamos en una sola tabla
tablas = []
for nombre_archivo in subido.keys():
    if nombre_archivo.lower().endswith(".csv"):
        tabla = pd.read_csv(nombre_archivo)
    else:
        tabla = pd.read_excel(nombre_archivo)
    tablas.append(tabla)

datos = pd.concat(tablas, ignore_index=True)
print(f"Se cargaron {len(subido)} archivo(s) con un total de {len(datos)} filas.")
datos.head()

### Revisemos la tabla

La tabla debería tener estas columnas (tal como las exportó el simulador):

- **Longitud (m)**: la longitud del péndulo que usaste.
- **Período medido (s)**: el tiempo que mediste con el cronómetro para una oscilación completa.
- **Período teórico (s)**: el valor calculado con la fórmula (solo de referencia, no lo usaremos para calcular *g* porque sería "hacer trampa").
- **Error (%)**: la diferencia entre tu medición y el valor teórico.

Nos quedamos solo con la longitud y el período que **tú mediste**.

In [ ]:
# Renombramos las columnas para trabajar más fácil
datos = datos.rename(columns={
    "Longitud (m)": "longitud",
    "Período medido (s)": "periodo"
})

datos = datos[["longitud", "periodo"]].dropna()
print(f"Tienes {len(datos)} mediciones en total.")
datos

## Paso 2️⃣: Promediar mediciones repetidas

Si mediste el período varias veces para la misma longitud (¡buena práctica científica!), promediamos esas mediciones para reducir el error humano del cronómetro. También calculamos cuánto varían tus mediciones (desviación estándar).

In [ ]:
resumen = datos.groupby("longitud")["periodo"].agg(["mean", "std", "count"]).reset_index()
resumen.columns = ["longitud", "periodo_promedio", "desviacion_estandar", "num_mediciones"]
resumen["desviacion_estandar"] = resumen["desviacion_estandar"].fillna(0)
resumen

## Paso 3️⃣: Grafiquemos Período vs. Longitud

In [ ]:
plt.figure(figsize=(8, 5))
plt.errorbar(resumen["longitud"], resumen["periodo_promedio"],
             yerr=resumen["desviacion_estandar"], fmt="o", capsize=5,
             color="#6c5ce7", ecolor="#e74c3c", markersize=8)
plt.xlabel("Longitud (m)")
plt.ylabel("Período (s)")
plt.title("Período del péndulo vs. Longitud")
plt.grid(alpha=0.3)
plt.show()

Notarán que la curva **no es una línea recta**: el período crece con la raíz cuadrada de la longitud, no de forma proporcional.

## Paso 4️⃣: Convertir la relación en una línea recta

La fórmula $T = 2\pi\sqrt{L/g}$ no da una línea recta si graficamos $T$ contra $L$. Pero si elevamos el período al cuadrado:

$$T^2 = \frac{4\pi^2}{g} \, L$$

¡Esto sí es una línea recta! Tiene la forma $y = m x + b$, donde:
- $y = T^2$
- $x = L$
- la pendiente $m = \dfrac{4\pi^2}{g}$

Si encontramos la pendiente de esta línea, podemos despejar $g$:

$$g = \frac{4\pi^2}{m}$$

In [ ]:
resumen["periodo_cuadrado"] = resumen["periodo_promedio"] ** 2

# Ajuste lineal: T^2 = m * L + b  (m = pendiente, b = intercepto)
m, b = np.polyfit(resumen["longitud"], resumen["periodo_cuadrado"], 1)

print(f"Pendiente (m): {m:.4f} s²/m")
print(f"Intercepto (b): {b:.4f} s²")

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(resumen["longitud"], resumen["periodo_cuadrado"], color="#6c5ce7", s=80, label="Datos experimentales")

x_linea = np.linspace(0, resumen["longitud"].max() * 1.1, 100)
y_linea = m * x_linea + b
plt.plot(x_linea, y_linea, color="#e74c3c", linewidth=2, label=f"Ajuste: T² = {m:.3f}·L + {b:.3f}")

plt.xlabel("Longitud (m)")
plt.ylabel("Período² (s²)")
plt.title("Linealización: T² vs. Longitud")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## Paso 5️⃣: Calculemos *g*

In [ ]:
g_experimental = 4 * np.pi**2 / m
g_aceptado = 9.81
error_porcentual = abs(g_experimental - g_aceptado) / g_aceptado * 100

print(f"🌍 Valor experimental de g: {g_experimental:.3f} m/s²")
print(f"📖 Valor aceptado de g:     {g_aceptado} m/s²")
print(f"📊 Error porcentual:        {error_porcentual:.2f}%")

## 🤔 Para reflexionar y discutir en clase

1. ¿Tu valor de *g* fue mayor, menor o parecido al valor aceptado (9.81 m/s²)? ¿A qué crees que se debe la diferencia?
2. ¿Qué fuentes de error existen al medir el período con un cronómetro y tu propio reflejo (tiempo de reacción)?
3. Si repitieras cada medición varias veces y promediaras, ¿esperarías un resultado más preciso? ¿Por qué?
4. La fórmula que usamos solo es válida para ángulos pequeños (menores a ~15-20°). ¿Qué pasaría si el péndulo se soltara desde un ángulo muy grande?
5. Si el intercepto $b$ no fuera exactamente cero, ¿qué podría estar indicando sobre el experimento?

## Extra: Guarda tu tabla de resultados (opcional)

Esta celda genera un archivo de Excel con tu tabla resumen (longitud, período promedio, desviación estándar) y lo descarga a tu computadora, para que lo incluyas en tu informe.

In [ ]:
resumen.to_excel("resumen_pendulo.xlsx", index=False)
files.download("resumen_pendulo.xlsx")

## 🎯 Desafío final: ¿A qué planeta trasladamos el péndulo?

Mientras no mirabas, movimos tu péndulo a otro lugar del sistema solar. Usa exactamente el mismo método que ya aprendiste (medir, promediar, linealizar y ajustar) para calcular la gravedad de ese lugar misterioso y descubrir dónde está.

Repite el experimento en este applet:

🔗 **https://cphysplus.github.io/fisica-interactiva/pendulo_planeta_misterioso.html**

1. Ajusta la longitud, suelta el péndulo, mide el período tú mismo para varias longitudes (varias repeticiones por longitud si puedes).
2. Presiona **"Exportar tabla a Excel"** y descarga el archivo.
3. Súbelo en la celda de abajo.

In [ ]:
print("Selecciona el archivo del planeta misterioso (.xlsx o .csv):")
subido2 = files.upload()

In [ ]:
# Leemos, combinamos y preparamos los datos del planeta misterioso
tablas2 = []
for nombre_archivo in subido2.keys():
    if nombre_archivo.lower().endswith(".csv"):
        tabla2 = pd.read_csv(nombre_archivo)
    else:
        tabla2 = pd.read_excel(nombre_archivo)
    tablas2.append(tabla2)

datos2 = pd.concat(tablas2, ignore_index=True)
datos2 = datos2.rename(columns={
    "Longitud (m)": "longitud",
    "Período medido (s)": "periodo"
})
datos2 = datos2[["longitud", "periodo"]].dropna()

resumen2 = datos2.groupby("longitud")["periodo"].agg(["mean", "std"]).reset_index()
resumen2.columns = ["longitud", "periodo_promedio", "desviacion_estandar"]
resumen2["periodo_cuadrado"] = resumen2["periodo_promedio"] ** 2
resumen2

In [ ]:
# Mismo ajuste lineal que antes: T^2 = m * L + b  ->  g = 4*pi^2 / m
m2, b2 = np.polyfit(resumen2["longitud"], resumen2["periodo_cuadrado"], 1)
g_experimental2 = 4 * np.pi**2 / m2

plt.figure(figsize=(8, 5))
plt.scatter(resumen2["longitud"], resumen2["periodo_cuadrado"], color="#6c5ce7", s=80, label="Datos del planeta misterioso")
x_linea2 = np.linspace(0, resumen2["longitud"].max() * 1.1, 100)
plt.plot(x_linea2, m2 * x_linea2 + b2, color="#e74c3c", linewidth=2, label=f"Ajuste: T² = {m2:.3f}·L + {b2:.3f}")
plt.xlabel("Longitud (m)")
plt.ylabel("Período² (s²)")
plt.title("Planeta misterioso: T² vs. Longitud")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

print(f"🪐 Gravedad calculada en el lugar misterioso: {g_experimental2:.2f} m/s²")

### ¿Con qué lugar del sistema solar coincide?

Antes de correr el código, busca y compara: esta es la gravedad superficial (aproximada) del Sol y los planetas del sistema solar.

| Cuerpo | Gravedad (m/s²) |
|---|---|
| Sol | 274.0 |
| Mercurio | 3.70 |
| Venus | 8.87 |
| Tierra | 9.81 |
| Marte | 3.71 |
| Júpiter | 24.79 |
| Saturno | 10.44 |
| Urano | 8.69 |
| Neptuno | 11.15 |

Con tu valor experimental de $g$, ¿a cuál de estos se parece más?

In [ ]:
gravedades = {
    "Sol": 274.0,
    "Mercurio": 3.70,
    "Venus": 8.87,
    "Tierra": 9.81,
    "Marte": 3.71,
    "Júpiter": 24.79,
    "Saturno": 10.44,
    "Urano": 8.69,
    "Neptuno": 11.15,
}

planeta_cercano = min(gravedades, key=lambda p: abs(gravedades[p] - g_experimental2))

print(f"Tu valor experimental de g: {g_experimental2:.2f} m/s²\n")
for cuerpo, valor in gravedades.items():
    marca = "  <-- tu resultado se parece más a este" if cuerpo == planeta_cercano else ""
    print(f"  {cuerpo:<12} g = {valor:>6.2f} m/s²{marca}")

print(f"\n🎉 ¡Trasladamos tu péndulo a: {planeta_cercano}!")